In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(os.path.abspath('')).resolve().parents[2]
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


import nvitk as nv
from nvitk import db

In [3]:
repo, xnat_config = db.get_repo_from_settings(return_xnat_config=True)

Using local root: ~/nvitk/dataset/nvitk-dataset


In [4]:
aux = repo.image(modality='4dflow', variables=['flow_mean'], wide=False)
aux = aux[(aux['variable_id' ] == 'flow_mean') & (aux['value_num'].notna())]

_SUBJECTS = pd.unique(aux.subject_uid)
_CLINICAL_VARS = ['age_at_mri', 'apoe_group', 'apoe', 'bmi', 'pp', 'map', 'hematocrit', 'pedframi10', 'pedframi30', 'sex', 'score2', 'tacsctot_group']
_PLAQUE_VARS = ['total_carotid_plaque_vol', 'total_femoral_plaque_vol', 'total_plaque_vol', 'right_carotid_plaque_vol', 'left_carotid_plaque_vol']
_COGNITIVE_VARS = ['z_compo_processingspeed', 'z_compo_globalfull', 'z_compo_epismemory', 'z_compo_attworkmemospeed']
_FLOW_VARS = ['flow_mean', 'pi']
_ASL_VARS = ['mean_cbf', 'att_mean']

In [5]:
clinical  = repo.clinical(variables=_CLINICAL_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
plaque    = repo.clinical(variables=_PLAQUE_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
cognitive = repo.cognitive(variables=_COGNITIVE_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
clinical   = repo.join([clinical, plaque, cognitive])
clinical


,subject_uid,visit_id_x,age_at_mri,apoe,apoe_group,bmi,hematocrit,pedframi10,pedframi30,pp,...,left_carotid_plaque_vol,right_carotid_plaque_vol,total_carotid_plaque_vol,total_femoral_plaque_vol,total_plaque_vol,visit_id,z_compo_attworkmemospeed,z_compo_epismemory,z_compo_globalfull,z_compo_processingspeed
0,PESA1006009,4,62.87,E3/E3,only_e3,27.968016,47.0,0.084716,0.272839,49.0,...,0.000000,0.000000,0.000000,33.320952,33.320952,4,-0.603836,0.126465,-0.438346,-1.714612
1,PESA10061584,4,64.27,E2/E3,any_e2,30.346074,48.0,0.078265,NaN,53.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,-0.443461,-1.858445,-1.086753,-0.679022
2,PESA10067929,4,64.54,E3/E4,any_e4,31.201777,42.0,0.15485,NaN,62.0,...,0.000000,10.015992,10.015992,535.628520,545.644512,4,0.215645,-0.623773,-0.050421,0.535598
3,PESA10086976,4,53.19,E3/E3,only_e3,27.053803,45.0,0.029603,0.150868,32.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,0.104212,0.632494,0.448210,-1.489313
4,PESA10112400,4,53.74,E3/E3,only_e3,26.612245,48.0,0.07328,0.292546,58.0,...,12.492144,0.000000,12.492144,0.000000,12.492144,4,0.091966,-0.247232,0.025983,-0.562022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,PESA9897316,4,55.35,E3/E3,only_e3,24.835646,52.0,0.23595,0.561753,39.0,...,0.000000,32.961096,32.961096,96.955488,129.916584,4,-1.146476,-2.453494,-2.027503,-1.471009
358,PESA9928801,4,62.13,E3/E3,only_e3,27.041644,45.0,0.047628,0.12778,31.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,1.524729,0.680927,1.275175,0.981430
359,PESA9935104,4,60.59,E3/E3,only_e3,19.896194,45.0,0.031566,0.071974,46.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,-0.294657,-1.795589,-1.474183,0.390962
360,PESA9947716,4,59.13,E3/E3,only_e3,29.757785,42.0,0.048233,0.186144,43.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,1.437316,0.566501,1.056801,1.191238


In [6]:
flow      = repo.image(modality='4dflow', variables=_FLOW_VARS, filters={'subject_uid': {'$in': _SUBJECTS}})
asl       = repo.image(modality='asl', variables=_ASL_VARS, atlas='vascular-8', filters={'subject_uid': {'$in': _SUBJECTS}})
image_df = repo.join([flow, asl])

territory_df = nv.stats.melt_imaging_territories(image_df, id_cols=['subject_uid'], flow_vars=_FLOW_VARS, asl_vars=_ASL_VARS, include_frame_index=False)
territory_df

,subject_uid,territory,modality_group,region_id,variable_id,value
0,PESA1006009,Posterior Circulation,flow,basilar,flow_mean,248.35062
1,PESA10061584,Posterior Circulation,flow,basilar,flow_mean,195.941298
2,PESA10067929,Posterior Circulation,flow,basilar,flow_mean,244.387905
3,PESA10086976,Posterior Circulation,flow,basilar,flow_mean,379.135876
4,PESA10112400,Posterior Circulation,flow,basilar,flow_mean,206.517756
...,...,...,...,...,...,...
18095,PESA9897316,Watershed,asl,watershed_8,mean_cbf,42.002036
18096,PESA9928801,Watershed,asl,watershed_8,mean_cbf,49.882072
18097,PESA9935104,Watershed,asl,watershed_8,mean_cbf,46.629408
18098,PESA9947716,Watershed,asl,watershed_8,mean_cbf,55.293241


In [7]:
analysis_df = clinical.copy()

for var in ['mean_cbf']:
    _long = territory_df[territory_df["variable_id"] == var].copy()
    _long["value"] = pd.to_numeric(_long["value"], errors="coerce")
    
    # 1. Rename the aggregated column to the specific variable name (e.g., 'pi', 'cbf', etc.)
    _agg = _long.groupby(["subject_uid", "territory"], as_index=False).agg(
        **{var: ("value", "mean")}
    )

    # 2. Merge on BOTH subject_uid and territory to keep the dataframe long/tidy
    if "territory" not in analysis_df.columns:
        analysis_df = analysis_df.merge(_agg, on="subject_uid", how="inner")
    else:
        analysis_df = analysis_df.merge(_agg, on=["subject_uid", "territory"], how="inner")

analysis_df.drop(columns=["visit_id_x"], inplace=True)
analysis_df.dropna(inplace=True)
analysis_df

,subject_uid,age_at_mri,apoe,apoe_group,bmi,hematocrit,pedframi10,pedframi30,pp,map,...,total_carotid_plaque_vol,total_femoral_plaque_vol,total_plaque_vol,visit_id,z_compo_attworkmemospeed,z_compo_epismemory,z_compo_globalfull,z_compo_processingspeed,territory,mean_cbf
0,PESA1006009,62.87,E3/E3,only_e3,27.968016,47.0,0.084716,0.272839,49.0,91.333333,...,0.0,33.320952,33.320952,4,-0.603836,0.126465,-0.438346,-1.714612,Anterior Circulation,57.134655
1,PESA1006009,62.87,E3/E3,only_e3,27.968016,47.0,0.084716,0.272839,49.0,91.333333,...,0.0,33.320952,33.320952,4,-0.603836,0.126465,-0.438346,-1.714612,Posterior Circulation,59.736089
2,PESA1006009,62.87,E3/E3,only_e3,27.968016,47.0,0.084716,0.272839,49.0,91.333333,...,0.0,33.320952,33.320952,4,-0.603836,0.126465,-0.438346,-1.714612,Watershed,50.179122
9,PESA10086976,53.19,E3/E3,only_e3,27.053803,45.0,0.029603,0.150868,32.0,89.666667,...,0.0,0.000000,0.000000,4,0.104212,0.632494,0.448210,-1.489313,Anterior Circulation,57.903620
10,PESA10086976,53.19,E3/E3,only_e3,27.053803,45.0,0.029603,0.150868,32.0,89.666667,...,0.0,0.000000,0.000000,4,0.104212,0.632494,0.448210,-1.489313,Posterior Circulation,52.097891
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1081,PESA9947716,59.13,E3/E3,only_e3,29.757785,42.0,0.048233,0.186144,43.0,85.333333,...,0.0,0.000000,0.000000,4,1.437316,0.566501,1.056801,1.191238,Posterior Circulation,50.044613
1082,PESA9947716,59.13,E3/E3,only_e3,29.757785,42.0,0.048233,0.186144,43.0,85.333333,...,0.0,0.000000,0.000000,4,1.437316,0.566501,1.056801,1.191238,Watershed,55.293241
1083,PESA9972964,53.65,E3/E3,only_e3,19.322185,41.0,0.013442,0.054825,35.0,85.666667,...,0.0,0.000000,0.000000,4,1.542101,0.905924,1.337689,1.756291,Anterior Circulation,57.358551
1084,PESA9972964,53.65,E3/E3,only_e3,19.322185,41.0,0.013442,0.054825,35.0,85.666667,...,0.0,0.000000,0.000000,4,1.542101,0.905924,1.337689,1.756291,Posterior Circulation,47.088127


Cacs

In [9]:
res, df_fit = nv.stats.fit_or_load_mixedlm(
    model_path=None,
    data=analysis_df,
    formula="mean_cbf ~ C(tacsctot_group, Treatment('g0')) * C(territory, Treatment('Anterior Circulation')) + age_at_mri + sex + hematocrit",
    groups="territory",
    re_formula="0",
    vc_formula={"subject": "0 + C(subject_uid)"},
    overwrite=False,
    required_columns=["territory", "tacsctot_group", "age_at_mri", "sex", "hematocrit", "subject_uid", "score2"],
    dropna_columns=["territory", "tacsctot_group", "age_at_mri", "sex", "hematocrit", "subject_uid", "score2"],
    fit_kwargs={"reml": True, "method": "lbfgs", "maxiter": 2000},
)

nv.stats.print_mixedlm_info(
    res,
    outcome_name="CACS - PI",
    group_name="territory",
    vc_group_name="subject_uid",
    output_path=None,
)


LinAlgError: Singular matrix

In [ ]:
fig = nv.stats.plot_mixedlm_params(
    result=res,
    df_fit=df_fit,
    x="tacsctot_group",
    y="mean_cbf",
    group="territory",
    mode="auto",
    include_points=True,
    output_path=None,
    title="CACS - PI mixed model",
    x_label="tacsctot_group",
    y_label="mean_cbf",
    covariate_refs={"sex": 0.0, "hematocrit": float(df_fit["hematocrit"].mean()), "score2": float(df_fit["score2"].mean())},
)
fig.clf()